# 📧 Plantilla: Extractor de Correos

## ¿Cuándo usar cada versión?

| Versión | Cuándo usarla | ¿Necesita contraseña? |
|---|---|---|
| **Versión A — Outlook (win32com)** | Tienes Outlook instalado en Windows y ya estás logueado | ❌ No (Windows ya autenticó) |
| **Versión B — IMAP** | Gmail, Hotmail, Yahoo, o cualquier correo sin Outlook | ✅ Sí |

---

## ⚠️ Importante sobre contraseñas

**Nunca escribas tu contraseña directamente en el código.**  
Usa variables de entorno o un archivo `.env` (ver instrucciones abajo).

---
# VERSIÓN A — Outlook (win32com)
### Requisitos: Windows + Microsoft Outlook instalado y con sesión activa
### NO necesita contraseña

In [ ]:
# ============================================================
# VERSIÓN A: Extractor de correos via Outlook (win32com)
# ============================================================
# pip install pywin32

import win32com.client
from datetime import datetime
import json
import os
import logging

# ─────────────────────────────────────────
# 🔧 CONFIGURA AQUÍ — solo cambia estas líneas
# ─────────────────────────────────────────
CORREO          = "tu.correo@empresa.com"   # ← tu dirección de correo en Outlook
CARPETA         = "Bandeja de entrada"       # ← carpeta que quieres leer
LIMITE_CORREOS  = 10                         # ← cuántos correos extraer (0 = todos)
RUTA_SALIDA     = r"C:\ruta\donde\guardar"   # ← dónde guardar el JSON
# ─────────────────────────────────────────

# Configurar logs
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("outlook_extraction.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger()

# Crear directorio de salida si no existe
os.makedirs(RUTA_SALIDA, exist_ok=True)
archivo_json = os.path.join(RUTA_SALIDA, "correos_outlook.json")

correos_data = []

try:
    logger.info("Conectando con Outlook...")
    outlook = win32com.client.Dispatch("Outlook.Application").GetNamespace("MAPI")

    # Verificar carpetas disponibles si hay error
    try:
        bandeja = outlook.Folders[CORREO].Folders[CARPETA]
        logger.info(f"Carpeta '{CARPETA}' encontrada")
    except Exception as e:
        carpetas_disponibles = [f.Name for f in outlook.Folders]
        logger.error(f"Error al acceder a la carpeta. Carpetas disponibles: {carpetas_disponibles}")
        raise

    messages = bandeja.Items
    messages.Sort("[ReceivedTime]", True)   # True = más recientes primero
    logger.info(f"Total mensajes en carpeta: {messages.Count}")

    fecha_extraccion = datetime.now().isoformat()
    count = 0

    for message in messages:
        try:
            correos_data.append({
                "ID"             : message.EntryID,
                "ConversationID" : getattr(message, "ConversationID", None),
                "Asunto"         : message.Subject or "(Sin Asunto)",
                "Remitente"      : message.SenderEmailAddress or "(Desconocido)",
                "FechaRecepcion" : message.ReceivedTime.isoformat() if hasattr(message, "ReceivedTime") else None,
                "Cuerpo"         : message.Body or "(Sin contenido)",
                "FechaExtraccion": fecha_extraccion
            })
            count += 1
            logger.info(f"[{count}] {message.Subject}")

            if LIMITE_CORREOS and count >= LIMITE_CORREOS:
                logger.info(f"Límite de {LIMITE_CORREOS} correos alcanzado")
                break
        except Exception as e:
            logger.error(f"Error en correo {count}: {e}")

    # Guardar JSON
    with open(archivo_json, "w", encoding="utf-8") as f:
        json.dump(correos_data, f, ensure_ascii=False, indent=4)
    logger.info(f"✅ {len(correos_data)} correos guardados en: {archivo_json}")

except Exception as e:
    logger.error(f"Error general: {e}")

print(f"\n✅ Proceso finalizado. Se extrajeron {len(correos_data)} correos.")

---
# VERSIÓN B — IMAP (Gmail, Hotmail, Yahoo, etc.)
### SÍ necesita contraseña — se lee desde variables de entorno

### 🔑 Cómo configurar la contraseña de forma segura

**Opción 1 — Variable de entorno (recomendada):**
```bash
# En Windows (cmd):
setx EMAIL_PASSWORD "tu_contraseña_aqui"

# En Mac/Linux:
export EMAIL_PASSWORD="tu_contraseña_aqui"
```

**Opción 2 — Archivo .env:**
Crea un archivo llamado `.env` en la misma carpeta con este contenido:
```
EMAIL_USER=tu.correo@gmail.com
EMAIL_PASSWORD=tu_contraseña
```
Luego instala: `pip install python-dotenv`

**Para Gmail:** necesitas crear una "Contraseña de aplicación" en  
https://myaccount.google.com/apppasswords (requiere 2FA activado)

In [ ]:
# ============================================================
# VERSIÓN B: Extractor de correos via IMAP
# Compatible con Gmail, Hotmail, Outlook.com, Yahoo, etc.
# ============================================================
# pip install python-dotenv

import imaplib
import email
from email.header import decode_header
from datetime import datetime
import json
import os
import logging
from dotenv import load_dotenv   # pip install python-dotenv

load_dotenv()  # carga variables del archivo .env

# ─────────────────────────────────────────
# 🔧 CONFIGURA AQUÍ — solo cambia estas líneas
# ─────────────────────────────────────────
SERVIDOR_IMAP  = "imap.gmail.com"            # ← ver tabla abajo
PUERTO         = 993                          # ← 993 para SSL (casi siempre)
CORREO         = os.getenv("EMAIL_USER",     "tu.correo@gmail.com")  # ← o escribe directo
CONTRASENA     = os.getenv("EMAIL_PASSWORD", "")                     # ← NUNCA escribir aquí
CARPETA        = "INBOX"                      # ← carpeta a leer
LIMITE_CORREOS = 10                           # ← cuántos correos extraer
RUTA_SALIDA    = "./data"                     # ← dónde guardar el JSON
# ─────────────────────────────────────────

# Servidores IMAP más comunes:
# Gmail:        imap.gmail.com
# Outlook/Hotmail: outlook.office365.com
# Yahoo:        imap.mail.yahoo.com
# Telefónica:   (consulta con IT, suele ser outlook.office365.com)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

os.makedirs(RUTA_SALIDA, exist_ok=True)
archivo_json = os.path.join(RUTA_SALIDA, "correos_imap.json")

def decodificar_texto(texto):
    """Decodifica asuntos y remitentes que vienen en distintos encodings."""
    if texto is None:
        return "(Sin contenido)"
    partes = decode_header(texto)
    resultado = ""
    for parte, encoding in partes:
        if isinstance(parte, bytes):
            resultado += parte.decode(encoding or "utf-8", errors="replace")
        else:
            resultado += parte
    return resultado

def extraer_cuerpo(msg):
    """Extrae el cuerpo del correo (texto plano)."""
    if msg.is_multipart():
        for parte in msg.walk():
            if parte.get_content_type() == "text/plain":
                return parte.get_payload(decode=True).decode(errors="replace")
    else:
        return msg.get_payload(decode=True).decode(errors="replace")
    return "(Sin contenido)"

correos_data = []

try:
    logger.info(f"Conectando a {SERVIDOR_IMAP}...")
    mail = imaplib.IMAP4_SSL(SERVIDOR_IMAP, PUERTO)
    mail.login(CORREO, CONTRASENA)
    logger.info("Conexión exitosa")

    mail.select(CARPETA)
    _, ids = mail.search(None, "ALL")
    lista_ids = ids[0].split()
    lista_ids = lista_ids[::-1]  # más recientes primero
    logger.info(f"Total mensajes encontrados: {len(lista_ids)}")

    fecha_extraccion = datetime.now().isoformat()
    count = 0

    for uid in lista_ids:
        try:
            _, datos = mail.fetch(uid, "(RFC822)")
            msg = email.message_from_bytes(datos[0][1])

            asunto   = decodificar_texto(msg["Subject"])
            remitente = decodificar_texto(msg["From"])
            fecha    = msg["Date"]
            cuerpo   = extraer_cuerpo(msg)

            correos_data.append({
                "ID"             : uid.decode(),
                "Asunto"         : asunto,
                "Remitente"      : remitente,
                "FechaRecepcion" : fecha,
                "Cuerpo"         : cuerpo[:2000],    # primeros 2000 caracteres
                "FechaExtraccion": fecha_extraccion
            })
            count += 1
            logger.info(f"[{count}] {asunto}")

            if LIMITE_CORREOS and count >= LIMITE_CORREOS:
                logger.info(f"Límite de {LIMITE_CORREOS} correos alcanzado")
                break
        except Exception as e:
            logger.error(f"Error en correo {uid}: {e}")

    mail.logout()

    with open(archivo_json, "w", encoding="utf-8") as f:
        json.dump(correos_data, f, ensure_ascii=False, indent=4)
    logger.info(f"✅ {len(correos_data)} correos guardados en: {archivo_json}")

except imaplib.IMAP4.error as e:
    logger.error(f"Error de autenticación IMAP: {e}")
    logger.error("Verifica: usuario, contraseña, y que IMAP esté habilitado en tu cuenta")
except Exception as e:
    logger.error(f"Error general: {e}")

print(f"\n✅ Proceso finalizado. Se extrajeron {len(correos_data)} correos.")

---
# 🔍 Ver los correos extraídos

In [ ]:
import json
import pandas as pd

# Cargar el JSON guardado (cambia la ruta según la versión usada)
with open("./data/correos_imap.json", encoding="utf-8") as f:  # ← cambia si usaste la versión A
    correos = json.load(f)

# Ver como tabla
df = pd.DataFrame(correos)
df[['Asunto', 'Remitente', 'FechaRecepcion']].head(10)

In [ ]:
# Ver el cuerpo de un correo específico
INDICE = 0   # ← cambia el número para ver otro correo

correo = correos[INDICE]
print(f"Asunto   : {correo['Asunto']}")
print(f"Remitente: {correo['Remitente']}")
print(f"Fecha    : {correo['FechaRecepcion']}")
print(f"\n--- CUERPO ---\n{correo['Cuerpo'][:500]}")